# Genome Repair Console -- Stage 0 on Kaggle

**Before running anything:** Settings (top right) -> Accelerator -> **GPU T4 x2** -> Internet -> **On**.

**Do NOT use P100** -- Kaggle's current PyTorch build has dropped support for the P100's older GPU architecture (confirmed via a real run: `CUDA error: no kernel image is available for execution on the device`). T4 uses a newer architecture that is supported.

**Add Data** (top right) -> attach your uploaded code dataset (see chat for what to include).

**If you've made any local code changes since you last uploaded** (e.g. the AMP/checkpoint-resume/confidence-score updates) -- re-zip and upload a NEW VERSION of your code dataset before running this notebook. Kaggle only sees whatever was in the dataset at upload time; it has no idea your local files changed.

**Important: "Save Version -> Save & Run All" does NOT continue your interactive session.** It re-runs the ENTIRE notebook from scratch in a brand new kernel -- it does not inherit your interactive session's progress. Cell 8 below auto-detects a LOCAL checkpoint (from an earlier invocation in the same session) as well as one from an attached prior-session dataset, so re-running cell 8 itself is now safe either way -- but committing a version is still a fresh run of every cell from the top.

## 1. Confirm the GPU Kaggle actually gave you

Should print a Tesla T4 (or two). If it prints P100, go back to Settings and switch the accelerator, then restart the session -- don't proceed.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 2. System dependencies (minimap2) -- needs Internet On, runs fine as root on Kaggle

In [2]:
!apt-get update -qq && apt-get install -y -qq minimap2
!minimap2 --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package minimap2.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../minimap2_2.24+dfsg-2_amd64.deb ...
Unpacking minimap2 (2.24+dfsg-2) ...
Setting up minimap2 (2.24+dfsg-2) ...
Processing triggers for man-db (2.10.2-1) ...
2.24-r1122


## 3. Python dependencies -- Badread pinned to the version requirements.txt specifies. Torch/fastapi already ship on Kaggle images; edlib/mappy typically need installing.

In [3]:
import importlib
def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
        print(f'{pkg}: already available')
    except ImportError:
        get_ipython().system(f'pip install -q {pip_name or pkg}')

ensure('edlib')
ensure('mappy')
ensure('fastapi')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.5/397.5 kB 8.2 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.0/144.0 kB 3.2 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fastapi: already available


In [4]:
!pip show badread 2>/dev/null | grep -q 'Version: 0.4.2' && echo 'badread 0.4.2 already installed' || pip install -q git+https://github.com/rrwick/Badread.git@v0.4.2
!badread --version


  Preparing metadata (setup.py) ... done
Badread v0.4.2


## 4. Copy your code into the writable working directory

`/kaggle/input/` is READ-ONLY -- training writes checkpoints and generated training pairs, so the project has to live under `/kaggle/working/`, not run directly from `/kaggle/input/`.

`CODE_DATASET_SLUG` below is set to what your last run actually confirmed works. If you re-upload as a new dataset, verify the correct path first with `!ls /kaggle/input/` and `!ls /kaggle/input/*/` before editing this.

In [5]:
CODE_DATASET_SLUG = 'datasets/syedmuksid/project-code-v5/project-code'  # confirmed working in your last run

import shutil, os
src = f'/kaggle/input/{CODE_DATASET_SLUG}'
dst = '/kaggle/working/final-year-project'

if os.path.exists(dst):
    print('Project already copied, skipping (delete /kaggle/working/final-year-project to force a fresh copy).')
else:
    shutil.copytree(src, dst)
    print(f'Copied {src} -> {dst}')

%cd /kaggle/working/final-year-project
!ls


Copied /kaggle/input/datasets/syedmuksid/project-code-v5/project-code -> /kaggle/working/final-year-project
/kaggle/working/final-year-project
backend       config.py  docs	   __init__.py	README.md	  tests
benchmarking  data	 frontend  model	requirements.txt  training


## 5. Sanity gate: syntax + the full test suite, before spending any GPU time

If this cell fails, STOP and fix it before running Stage 0 -- don't burn GPU hours on a broken environment.

In [6]:
!find . -name '*.py' -exec python -m py_compile {} \;
!python -m pytest tests/ -q


..............................                                           [100%]
30 passed in 4.44s


## 6. Confirm the reference genome landed where config.py expects it

`config.py` hardcodes `data/reference/ecoli_k12_mg1655.fasta` (NC_000913.3) as a path relative to the project root -- since we copied the whole project (including `data/reference/`) into `/kaggle/working/final-year-project` above, this should already just work with no symlink needed. This cell only double-checks it, so a bad upload fails loudly here instead of silently mid-training.

In [7]:
from pathlib import Path
ref = Path('data/reference/ecoli_k12_mg1655.fasta')
assert ref.exists(), f'{ref} not found -- did you include data/reference/ in your uploaded code dataset?'
print(f'{ref}: {ref.stat().st_size:,} bytes -- OK')


data/reference/ecoli_k12_mg1655.fasta: 4,708,035 bytes -- OK


## 7. Generate training data (Stage 0, part 1) -- reuses a cached copy if you have one

This is CPU-only and takes 40+ minutes at `--quantity 50x`, but it's fully deterministic (fixed seed + quantity), so there's no reason to regenerate it every session. First time: leave `TRAINING_DATA_DATASET_SLUG = None`, let it generate, then follow the instructions printed at the end to save it as a dataset. Every session after that: set the slug and it copies instantly instead of regenerating.

In [8]:
TRAINING_DATA_DATASET_SLUG = None

import shutil, subprocess
from pathlib import Path

target = Path('data/training_pairs.jsonl')

if target.exists() and target.stat().st_size > 0:
    print(f'Found existing training data in project folder ({target.stat().st_size:,} bytes) -- skipping generation.')
    reused_cache = True
else:
    reused_cache = False

if not reused_cache:
    subprocess.run([
        'python', '-m', 'data.simulator',
        '--reference', 'data/reference/ecoli_k12_mg1655.fasta',
        '--output', str(target), '--quantity', '50x',
    ], check=True)
    print(
        '\nTo avoid regenerating this every session: Save Version -> Save & Run All (Commit) -> '
        'once done, open that version\'s Output tab -> Create Dataset from data/training_pairs.jsonl. '
        'Then set TRAINING_DATA_DATASET_SLUG above to that new dataset\'s slug in future sessions.'
    )


Found existing training data in project folder (619,448,048 bytes) -- skipping generation.


## 8. Train, with Kaggle's 12-hour limit built in

This cell auto-detects a resume source, checked in this order:

1. **A local `latest.training_state.pt`** already sitting in `/kaggle/working/checkpoints/` from an earlier invocation of THIS cell in the same session (e.g. you re-ran the cell, or a transient error caused a silent restart). This is new -- it's exactly what was missing last time, when a run reached step 1030+ and then silently restarted from step 0 with no resume, because only the cross-session path (below) was ever checked.
2. **A previous session's saved output**, if you attach it as a second input dataset and set `PREV_SESSION_DATASET_SLUG`.

If neither exists, it starts fresh.

`--max-wall-time-seconds` is set to 11.5 hours (leaving 30 min margin beyond the code's own internal 5-minute safety margin) -- the run saves and exits cleanly on its own terms well before Kaggle's hard kill.

`--log-every 100` -- was hardcoded at 10 before (one line every 10 steps), which is a lot of lines against Kaggle's log buffer over 128,000+ total steps. Raised to 100 here; requires the updated `train.py` with `--log-every` exposed as a real CLI flag -- make sure that file is in your uploaded code dataset.

Mixed precision (AMP) is enabled automatically on a CUDA device -- no flag needed.

**Expect this to span multiple sessions.** ~128,000 total steps is many hours even on a real GPU -- that's expected, not a problem.

In [9]:
PREV_SESSION_DATASET_SLUG = 'checkpoints-v1'  # <-- set to e.g. 'my-notebook-output-v1' once you have a prior session to resume from

import subprocess, os
from pathlib import Path

CHECKPOINT_PATH = '/kaggle/working/checkpoints/model_best.pt'
TRAINING_STATE_PATH = '/kaggle/working/checkpoints/latest.training_state.pt'

# Mitigates CUDA allocator fragmentation (PyTorch's own suggestion from the
# earlier OOM error message) -- cheap to set, complements --max-chunk-size.
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

resume_args = []

# 1) LOCAL checkpoint first -- covers the case this cell (or the whole
#    notebook) gets re-run within the same session, which previously
#    silently discarded all progress since nothing checked for this.
if Path(TRAINING_STATE_PATH).exists():
    resume_args = ['--resume-from', TRAINING_STATE_PATH]
    print(f'Found a LOCAL training state at {TRAINING_STATE_PATH} -- resuming from it.')
elif PREV_SESSION_DATASET_SLUG:
    # 2) Cross-session resume via an attached prior-session output dataset.
    prev_state = f'/kaggle/input/{PREV_SESSION_DATASET_SLUG}/latest.training_state.pt'
    if os.path.exists(prev_state):
        resume_args = ['--resume-from', prev_state]
        print(f'Resuming from previous session: {prev_state}')
    else:
        print(f'WARNING: PREV_SESSION_DATASET_SLUG is set but {prev_state} does not exist -- starting fresh.')
else:
    print('No local or previous-session training state found -- starting fresh.')

cmd = [
    'python', '-m', 'training.train',
    '--train-data', 'data/training_pairs.jsonl',
    '--epochs', '5',
    '--batch-size', '32',
    '--chunk-size', '256',
    '--max-chunk-size', '256',
    '--checkpoint-path', CHECKPOINT_PATH,
    '--training-state-path', TRAINING_STATE_PATH,
    '--checkpoint-every-steps', '200',
    '--max-wall-time-seconds', str(11.5 * 3600),
    '--log-every', '100',
] + resume_args

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)


No local or previous-session training state found -- starting fresh.
Running: python -m training.train --train-data data/training_pairs.jsonl --epochs 5 --batch-size 32 --chunk-size 256 --max-chunk-size 256 --checkpoint-path /kaggle/working/checkpoints/model_best.pt --training-state-path /kaggle/working/checkpoints/latest.training_state.pt --checkpoint-every-steps 200 --max-wall-time-seconds 41400.0 --log-every 100


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/final-year-project/training/train.py", line 455, in <module>
    train(
  File "/kaggle/working/final-year-project/training/train.py", line 319, in train
    output = model(
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/final-year-project/model/sequence_translation_model.py", line 141, in forward
    logits, predicted_tokens, attention_matrix = self.decoder(
                                                 ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.1

Training on GPU: Tesla T4
Mixed precision (AMP): enabled
Config: batch_size=32, chunk_size=256, epochs=5. NOTE: the decoder is autoregressive -- per-batch cost scales with chunk_size (more sequential decode steps), largely independent of GPU vs CPU. If training feels slow regardless of device, a smaller --chunk-size and/or larger --batch-size (to amortize fixed per-step loop overhead) will help more than switching hardware alone.
Training pairs (reads): 13547 | Validation pairs (reads): 1505
GenomeCorrectionDataset: 13547 raw pair(s) -> 820914 chunk(s) (chunk_size=256, min_chunk_size=16). 0 kept chunk(s) still exceed chunk_size (single alignment segment larger than chunk_size). 106747 chunk(s) dropped for exceeding max_chunk_size=256. 85 chunk(s) dropped for having a clean/target side shorter than min_chunk_size (pure-insertion segments -- noisy side long enough, clean side too short or empty). 5 raw pair(s) produced no usable chunks.
GenomeCorrectionDataset: 1505 raw pair(s) -> 90691 

KeyboardInterrupt: 

## 9. If the session ended before training finished: how to continue next time

1. Click **Save Version** (top right) -> **Save & Run All (Commit)**. Wait for it to finish -- this attaches everything under `/kaggle/working/` (including `checkpoints/latest.training_state.pt`) as this notebook version's Output. Remember: this is a FRESH run from the top, not a continuation of your interactive session -- cell 8 will only find local progress from this fresh run's own invocations, not your earlier interactive one.
2. Open a **new session** of this same notebook.
3. **Add Data** -> **Your Work** -> **Notebook Output Files** -> select the version you just saved. This mounts it read-only at `/kaggle/input/<this-notebook-slug>/`.
4. In the training cell above, set `PREV_SESSION_DATASET_SLUG` to that notebook's slug, then re-run the notebook from the top.

The training cell will detect the saved `latest.training_state.pt`, restore model weights + optimizer state + GradScaler state + exact step position, and continue the teacher-forcing schedule from where it left off -- not from scratch.